## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [35]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

# Publications markdown generator

Takes a .bib of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

In [ ]:
import os
import calendar
from pybtex.database import parse_file

bib_data = parse_file('publications.bib')
#print(bib_data)

for entry in bib_data.entries:
# create the name of each md file
    year  = bib_data.entries[entry].fields['year']
    if 'month' in bib_data.entries[entry].fields:
        month = list(calendar.month_abbr).index(bib_data.entries[entry].fields['month'].capitalize())
    else:
        month = 1
    pub_date = str(year) + "-" + str(month) + "-" + "1"
    
    md_filename   = pub_date + "-" + entry + ".md"
    #print(md_filename)

# cp pdf files to ../files and set up the link
    if 'file' in bib_data.entries[entry].fields:
        path = '/mnt/c' + bib_data.entries[entry].fields['file'][15:-4]
        path = path.replace(" ",    "\ ")
        path = path.replace("{\_}", "\_")
        os.system('cp ' + path + ' ../files')
        html_filename = '/files/' + os.path.basename(path).replace("\\", "")
    else:
        html_filename = ''
    #print(html_filename)

# set up author names
    num_author=len(bib_data.entries[entry].persons['author'])
    for count, author in enumerate(bib_data.entries[entry].persons['author']):
        if count == 0:
            author_list = author.last_names[0] + ", "
            for cc, first in enumerate(author.bibtex_first_names):
                if cc <len(author.bibtex_first_names)-1:
                    author_list += first + " "
                else:
                    author_list += first
        elif count < num_author-1:
            author_list += ", " 
            for cc, first in enumerate(author.bibtex_first_names):
                if cc <len(author.bibtex_first_names)-1:
                    author_list += first + " "
                else:
                    author_list += first
            author_list += " " + author.last_names[0]
        else:
            author_list += " and " 
            for cc, first in enumerate(author.bibtex_first_names):
                if cc <len(author.bibtex_first_names)-1:
                    author_list += first + " "
                else:
                    author_list += first
            author_list += " " + author.last_names[0]
        author_list = author_list.replace("{","")
        author_list = author_list.replace("}","")
    #print(author_list)

# set up the citation after the journal names
    citation = "<i>" + bib_data.entries[entry].fields['journal'] + "</i>"
    if 'volume' in bib_data.entries[entry].fields:
        citation += ", " + bib_data.entries[entry].fields['volume']
    if 'pages' in bib_data.entries[entry].fields:
        citation += ", " + bib_data.entries[entry].fields['pages']
    citation +=  ", doi:" + bib_data.entries[entry].fields['doi']
    citation += "." 
#    print(citation)
    
    ## YAML variables
    
    md = "---\ntitle: \"" + html_escape(bib_data.entries[entry].fields['title'][1:-1]) + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: """ + html_filename

    md += "\nyear: " + str(year) 

    md += "\nauthor: " + str(author_list) 

    md += "\nvenue: '" + html_escape(bib_data.entries[entry].fields['journal']) + "'"
    
    md += "\npaperurl: '" + bib_data.entries[entry].fields['url'] + "'"
    
    md += "\ncitation: '" + html_escape(citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)
#    print(md)
    

# Publications markdown generator

Takes a CSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

Output the Mendeley in .bib format first, and convert it to .csv using JabRef


In [43]:
!cat publications.csv

BibliographyType,ISBN,Identifier,Author,Title,Journal,Volume,Number,Month,Pages,Year,Address,Note,URL,Booktitle,Chapter,Edition,Series,Editor,Publisher,ReportType,Howpublished,Institution,Organizations,School,Annote,Custom1,Custom2,Custom3,Custom4,Custom5
7,"","Burrows2015","Burrows, D.  Alex; Chen, Gang; Sun, Lantao","Barotropic and Baroclinic Eddy Feedbacks in the Midlatitude Jet Variability and Responses to Climate Change–Like Thermal Forcings","Journal of the Atmospheric Sciences",74,1,"jan","111--132",2017,"","","http://journals.ametsoc.org/doi/10.1175/JAS-D-16-0047.1","","","","","","","","","","","","","","","","",""
7,"","Chen2007a","Chen, Gang; Held, Isaac M.; Robinson, Walter A.","Sensitivity of the Latitude of the Surface Westerlies to Surface Friction","Journal of the Atmospheric Sciences",64,8,"aug","2899--2915",2007,"","","http://journals.ametsoc.org/doi/abs/10.1175/JAS3995.1","","","","","","","","","","","","From Duplicate 2 (Sensitivity of the Latitude of the Surface W

## Import pandas

We are using the very handy pandas library for dataframes.

In [16]:
import pandas as pd

In [44]:
publications = pd.read_csv("publications.csv", sep=",", header=0)
publications

,BibliographyType,ISBN,Identifier,Author,Title,Journal,Volume,Number,Month,Pages,...,Howpublished,Institution,Organizations,School,Annote,Custom1,Custom2,Custom3,Custom4,Custom5
0,7,NaN,Burrows2015,"Burrows, D. Alex; Chen, Gang; Sun, Lantao",Barotropic and Baroclinic Eddy Feedbacks in th...,Journal of the Atmospheric Sciences,74.0,1,jan,111--132,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,NaN,Chen2007a,"Chen, Gang; Held, Isaac M.; Robinson, Walter A.",Sensitivity of the Latitude of the Surface Wes...,Journal of the Atmospheric Sciences,64.0,8,aug,2899--2915,...,NaN,NaN,NaN,NaN,From Duplicate 2 (Sensitivity of the Latitude ...,NaN,NaN,NaN,NaN,NaN
2,7,NaN,Chen2007b,"Chen, Gang; Held, Isaac M.",Phase speed spectra and the recent poleward sh...,Geophysical Research Letters,34.0,21,nov,L21805,...,NaN,NaN,NaN,NaN,From Duplicate 2 ( Phase speed spectra and the...,NaN,NaN,NaN,NaN,NaN
3,7,NaN,Chen2008,"Chen, Gang; Lu, Jian; Frierson, Dargan M. W.",Phase Speed Spectra and the Latitude of Surfac...,Journal of Climate,21.0,22,nov,5942--5959,...,NaN,NaN,NaN,NaN,From Duplicate 2 (Phase Speed Spectra and the ...,NaN,NaN,NaN,NaN,NaN
4,7,NaN,Chen2008b,"Chen, Gang; Zurita-Gotor, Pablo",The Tropospheric Jet Response to Prescribed Zo...,Journal of the Atmospheric Sciences,65.0,7,jul,2254--2271,...,NaN,NaN,NaN,NaN,From Duplicate 1 (The tropospheric jet respons...,NaN,NaN,NaN,NaN,NaN
5,7,NaN,Chen2009,"Chen, Gang; Plumb, R. Alan",Quantifying the Eddy Feedback and the Persiste...,Journal of the Atmospheric Sciences,66.0,12,dec,3707--3720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,NaN,Chen2010,"Chen, Gang; Plumb, R. Alan; Lu, Jian",Sensitivities of zonal mean atmospheric circul...,Geophysical Research Letters,37.0,12,jun,NaN,...,NaN,NaN,NaN,NaN,in press,NaN,NaN,NaN,NaN,NaN
7,7,NaN,Chen2011,"Chen, Gang; Ming, Yi; Singer, Noah D.; Lu, Jian",Testing the Clausius-Clapeyron constraint on t...,Geophysical Research Letters,38.0,NaN,feb,L04807,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,7,NaN,Chen2011c,"Chen, Gang; Sun, Lantao",Mechanisms of the Tropical Upwelling Branch of...,Journal of the Atmospheric Sciences,68.0,12,dec,2878--2892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,7,NaN,Chen2013c,"Chen, Gang; Lu, Jian; Sun, Lantao",Delineating the Eddy–Zonal Flow Interaction in...,Journal of the Atmospheric Sciences,70.0,7,jul,2214--2233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [45]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [46]:
publications

,BibliographyType,ISBN,Identifier,Author,Title,Journal,Volume,Number,Month,Pages,...,Howpublished,Institution,Organizations,School,Annote,Custom1,Custom2,Custom3,Custom4,Custom5
0,7,NaN,Burrows2015,"Burrows, D. Alex; Chen, Gang; Sun, Lantao",Barotropic and Baroclinic Eddy Feedbacks in th...,Journal of the Atmospheric Sciences,74.0,1,jan,111--132,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,NaN,Chen2007a,"Chen, Gang; Held, Isaac M.; Robinson, Walter A.",Sensitivity of the Latitude of the Surface Wes...,Journal of the Atmospheric Sciences,64.0,8,aug,2899--2915,...,NaN,NaN,NaN,NaN,From Duplicate 2 (Sensitivity of the Latitude ...,NaN,NaN,NaN,NaN,NaN
2,7,NaN,Chen2007b,"Chen, Gang; Held, Isaac M.",Phase speed spectra and the recent poleward sh...,Geophysical Research Letters,34.0,21,nov,L21805,...,NaN,NaN,NaN,NaN,From Duplicate 2 ( Phase speed spectra and the...,NaN,NaN,NaN,NaN,NaN
3,7,NaN,Chen2008,"Chen, Gang; Lu, Jian; Frierson, Dargan M. W.",Phase Speed Spectra and the Latitude of Surfac...,Journal of Climate,21.0,22,nov,5942--5959,...,NaN,NaN,NaN,NaN,From Duplicate 2 (Phase Speed Spectra and the ...,NaN,NaN,NaN,NaN,NaN
4,7,NaN,Chen2008b,"Chen, Gang; Zurita-Gotor, Pablo",The Tropospheric Jet Response to Prescribed Zo...,Journal of the Atmospheric Sciences,65.0,7,jul,2254--2271,...,NaN,NaN,NaN,NaN,From Duplicate 1 (The tropospheric jet respons...,NaN,NaN,NaN,NaN,NaN
5,7,NaN,Chen2009,"Chen, Gang; Plumb, R. Alan",Quantifying the Eddy Feedback and the Persiste...,Journal of the Atmospheric Sciences,66.0,12,dec,3707--3720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,NaN,Chen2010,"Chen, Gang; Plumb, R. Alan; Lu, Jian",Sensitivities of zonal mean atmospheric circul...,Geophysical Research Letters,37.0,12,jun,NaN,...,NaN,NaN,NaN,NaN,in press,NaN,NaN,NaN,NaN,NaN
7,7,NaN,Chen2011,"Chen, Gang; Ming, Yi; Singer, Noah D.; Lu, Jian",Testing the Clausius-Clapeyron constraint on t...,Geophysical Research Letters,38.0,NaN,feb,L04807,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,7,NaN,Chen2011c,"Chen, Gang; Sun, Lantao",Mechanisms of the Tropical Upwelling Branch of...,Journal of the Atmospheric Sciences,68.0,12,dec,2878--2892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,7,NaN,Chen2013c,"Chen, Gang; Lu, Jian; Sun, Lantao",Delineating the Eddy–Zonal Flow Interaction in...,Journal of the Atmospheric Sciences,70.0,7,jul,2214--2233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
import os
import calendar
import numpy
for row, item in publications.iterrows():

    year = item.Year
    if pd.isnull(item.Month): 
        month = 1
    else:
        month = list(calendar.month_abbr).index(item.Month.capitalize())
    
    pub_date = str(year) + "-" + str(month) + "-" + "1"
    md_filename   = pub_date + "-" + item.Identifier + ".md"
    html_filename = pub_date + "-" + item.Identifier
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.Title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename

    md += "\ndate: " + str(pub_date) 
    
    md += "\nvenue: '" + html_escape(item.Journal) + "'"
    
    if len(str(item.URL)) > 5:
        md += "\npaperurl: '" + item.URL + "'"
    
    md += "\ncitation: "
    
    md += "\n---"

    md_filename = os.path.basename(md_filename)
    
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

Barotropic and Baroclinic Eddy Feedbacks in the Midlatitude Jet Variability and Responses to Climate Change–Like Thermal Forcings
Sensitivity of the Latitude of the Surface Westerlies to Surface Friction
Phase speed spectra and the recent poleward shift of Southern Hemisphere surface westerlies
Phase Speed Spectra and the Latitude of Surface Westerlies: Interannual Variability and Global Warming Trend
The Tropospheric Jet Response to Prescribed Zonal Forcing in an Idealized Atmospheric Model
Quantifying the Eddy Feedback and the Persistence of the Zonal Index in an Idealized Atmospheric Model
Sensitivities of zonal mean atmospheric circulation to SST warming in an aqua-planet model
Testing the Clausius-Clapeyron constraint on the aerosol-induced changes in mean and extreme precipitation
Mechanisms of the Tropical Upwelling Branch of the Brewer–Dobson Circulation: The Role of Extratropical Waves
Delineating the Eddy–Zonal Flow Interaction in the Atmospheric Circulation Response to Clima

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

These files are in the publications directory, one directory below where we're working from.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.Year) + "-" + item.Title + ".md"
    html_filename = str(item.pub_date) + "-" + item.Title
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

In [38]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md  2010-10-01-paper-title-number-2.md


In [39]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

---
title: "Paper Title Number 1"
collection: publications
permalink: /publication/2009-10-01-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 2009-10-01
venue: 'Journal 1'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).